In [1]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [ ]:
schema_Bases = [
        bigquery.SchemaField("ID", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_OPERACION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("CERTIFICADO_BANCO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("NUMERO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONTO_LA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("ESTADO_LA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_EMISION_LA", bigquery.enums.SqlTypeNames.DATE)       
]

### FCR1

In [77]:
df_FCR1= pd.read_excel("C:/data/MEMOS CONTABLES/BRIMAC_FCR1_CONTROL_ENERO_2024-2026 ÚLTIMO.xlsx", sheet_name=0, dtype=str)
df_FCR1.columns = (df_FCR1.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

In [79]:
df_FCR1= df_FCR1.rename(columns={'SITE':'ID', 'FECHA_INGRESO':'FECHA_OPERACION', 'N_MERO_DE_CONTRATO_DE_SEGURO':'CERTIFICADO_BANCO', 
                                 'TIPO_DE_SEGURO':'PRODUCTO', 'IMPORTE_ORIGINAL':'MONTO', 'DIVISA_ORIGINAL':'MONEDA', 
                                 'LA':'NUMERO_LA', 'IMPORTE_DE_LA_GENERADA':'MONTO_LA', 'ESTADO_DE__LA_':'ESTADO_LA', 
                                 'FECHA_DE_EMISION_LA':'FECHA_EMISION_LA'})

In [80]:
df_FCR1['MONEDA'].value_counts()

MONEDA
Soles      27758
Dólares    10402
Name: count, dtype: int64

In [81]:
df_FCR1['PRODUCTO'] = df_FCR1['PRODUCTO'].str.upper()
df_FCR1['MONTO'] = df_FCR1['MONTO'].replace({',': ''}, regex=True)
df_FCR1['MONTO'] = pd.to_numeric(df_FCR1['MONTO'], errors="coerce").astype('float64')
df_FCR1['FECHA_OPERACION'] = pd.to_datetime(df_FCR1['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_FCR1['FECHA_EMISION_LA'] = pd.to_datetime(df_FCR1['FECHA_EMISION_LA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_FCR1.loc[df_FCR1['MONEDA'].fillna('').str.strip().str.lower().isin(['soles','sol']), 'MONEDA'] = 'SOLES'
df_FCR1.loc[df_FCR1['MONEDA'].fillna('').str.strip().str.lower().isin(['dólares', 'dolares']), 'MONEDA'] = 'DOLARES'

In [82]:
df_FCR1= df_FCR1[['ID', 'FECHA_OPERACION', 'CERTIFICADO_BANCO','PRODUCTO','MONEDA','MONTO', 'NUMERO_LA', 
                  'MONTO_LA', 'ESTADO_LA', 'FECHA_EMISION_LA']]

In [83]:
df_FCR1.head(3)

,ID,FECHA_OPERACION,CERTIFICADO_BANCO,PRODUCTO,MONEDA,MONTO,NUMERO_LA,MONTO_LA,ESTADO_LA,FECHA_EMISION_LA
0,21665,2024-01-01,00110178184000382668,SALUD A TU ALCANCE,SOLES,1138.0,166747568,1137.99,ABONADO,2024-01-12
1,21668,2024-01-02,00110183164000988164,PROTECCIÓN DE TARJETA,DOLARES,41.0,166852029,41,ABONADO,2024-01-17
2,21669,2024-01-02,00110194874000475663,MULTIRIESGO NEGOCIO,SOLES,2294.0,166852554,2293.99,ABONADO,2024-01-17


### FCR2

In [54]:
df_FCR2= pd.read_excel("C:/data/MEMOS CONTABLES/DEVOLUCION_A_PRORRATA_FCR2_OPERACIONES actualizado 3 (30) (1).xlsx", sheet_name=0, dtype=str)
df_FCR2.columns = (df_FCR2.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

In [56]:
df_FCR2= df_FCR2.drop(columns=['PRODUCTO'])

In [57]:
df_FCR2= df_FCR2.rename(columns={'N__CORRELATIVO___REQUERIMIENTO':'ID', 'FECHA_SOLICITUD_OPERACIONES':'FECHA_OPERACION', 'N__CONTRATO':'CERTIFICADO_BANCO', 
                                 'TIPO_SEGURO':'PRODUCTO', 'IMPORTE_ABONADO_AL_CLIENTE':'MONTO', 'DIVISA_ORIGINAL_PRIMA':'MONEDA', 
                                 'LA':'NUMERO_LA', 'IMPORTE':'MONTO_LA', 'F_EMISION_LA':'FECHA_EMISION_LA'})

In [63]:
df_FCR2['MONEDA'].value_counts()

MONEDA
DOLARES    16897
SOLES      14594
Name: count, dtype: int64

In [59]:
df_FCR2['PRODUCTO'] = df_FCR2['PRODUCTO'].str.upper()
df_FCR2['MONTO'] = df_FCR2['MONTO'].replace({',': ''}, regex=True)
df_FCR2['MONTO'] = pd.to_numeric(df_FCR2['MONTO'], errors="coerce").astype('float64')
df_FCR2['FECHA_OPERACION'] = pd.to_datetime(df_FCR2['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_FCR2['FECHA_EMISION_LA'] = pd.to_datetime(df_FCR2['FECHA_EMISION_LA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_FCR2.loc[df_FCR2['MONEDA'].fillna('').str.strip().str.lower().isin(['soles','sol']), 'MONEDA'] = 'SOLES'
df_FCR2.loc[df_FCR2['MONEDA'].fillna('').str.strip().str.lower().isin(['dólares', 'dolares']), 'MONEDA'] = 'DOLARES'

In [60]:
df_FCR2= df_FCR2[['ID', 'FECHA_OPERACION', 'CERTIFICADO_BANCO','PRODUCTO','MONEDA','MONTO', 'NUMERO_LA', 
                  'MONTO_LA', 'ESTADO_LA', 'FECHA_EMISION_LA']]

In [62]:
df_FCR2.head(3)

,ID,FECHA_OPERACION,CERTIFICADO_BANCO,PRODUCTO,MONEDA,MONTO,NUMERO_LA,MONTO_LA,ESTADO_LA,FECHA_EMISION_LA
0,42273,2023-06-22,00110235904001894351,VEHICULAR OPTATIVO,DOLARES,2958.43,NaN,NaN,NaN,NaT
1,42388,2023-06-23,00110285444002031147,SALUD A TU ALCANCE,SOLES,324.01,162523661,324.01,ACT,2023-07-07
2,42554,2023-06-26,00110976884000461892,SALUD A TU ALCANCE,SOLES,510.52,162528075,510.52,ACT,2023-07-07


### RECLAMOS

In [69]:
df_RECLAMOS= pd.read_excel("C:/data/MEMOS CONTABLES/CUADRO_OBLIGACIONES_BBVA_2026 (7).xlsx", sheet_name=0, dtype=str)
df_RECLAMOS.columns = (df_RECLAMOS.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))

In [70]:
df_RECLAMOS= df_RECLAMOS.rename(columns={'CASO_SF':'ID', 'FECHA_DE_SOLICITUD_DE_OPERACIONES':'FECHA_OPERACION', 
                                         'NRO_CONTRATO__CERTIFICADO_BANCO_':'CERTIFICADO_BANCO', 
                                         'MONTO_RECLAMADO_S__USD':'MONTO', 'TIPO_DE_MONEDA_DE_LA_CUENTA_BANCARIA':'MONEDA',
                                         'MTO_LA':'MONTO_LA', 'F_EMISION_LA':'FECHA_EMISION_LA'})

In [71]:
df_RECLAMOS['PRODUCTO'] = df_RECLAMOS['PRODUCTO'].str.upper()
df_RECLAMOS['MONTO'] = df_RECLAMOS['MONTO'].replace({',': ''}, regex=True)
df_RECLAMOS['MONTO'] = pd.to_numeric(df_RECLAMOS['MONTO'], errors="coerce").astype('float64')
df_RECLAMOS['FECHA_OPERACION'] = pd.to_datetime(df_RECLAMOS['FECHA_OPERACION'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_RECLAMOS['FECHA_EMISION_LA'] = pd.to_datetime(df_RECLAMOS['FECHA_EMISION_LA'],format='%Y-%m-%d %H:%M:%S', errors='coerce').dt.date
df_RECLAMOS.loc[df_RECLAMOS['MONEDA'].fillna('').str.strip().str.lower().isin(['soles','sol']), 'MONEDA'] = 'SOLES'
df_RECLAMOS.loc[df_RECLAMOS['MONEDA'].fillna('').str.strip().str.lower().isin(['usd', 'dolares']), 'MONEDA'] = 'DOLARES'

In [72]:
df_RECLAMOS['MONEDA'].value_counts()

MONEDA
SOLES      5260
DOLARES    1273
Name: count, dtype: int64

In [74]:
df_RECLAMOS= df_RECLAMOS[['ID', 'FECHA_OPERACION', 'CERTIFICADO_BANCO','PRODUCTO','MONEDA','MONTO', 'NUMERO_LA', 
                            'MONTO_LA', 'ESTADO_LA', 'FECHA_EMISION_LA']]

In [75]:
df_RECLAMOS.head(3)

,ID,FECHA_OPERACION,CERTIFICADO_BANCO,PRODUCTO,MONEDA,MONTO,NUMERO_LA,MONTO_LA,ESTADO_LA,FECHA_EMISION_LA
0,0027055041,2026-01-05,00110354824000294250,VEHICULAR OPTATIVO,DOLARES,870.45,178316796,870.45,ACT,2026-01-08
1,0027055040,2026-01-05,00110716814000247004,SALUD A TU ALCANCE,SOLES,1991.00,178375156,1990.97,ACT,2026-01-14
2,0027055017,2026-01-05,00110333264000574828,HOGAR TOTAL,SOLES,1553.80,178315151,1553.8,ACT,2026-01-08
